# Feature Extraction

This is where raw text becomes numbers the models can actually learn from.

We compute five feature groups per response: token length, lexical diversity (type-token ratio), readability (Flesch score), overlap with the reference (ROUGE-L), and semantic similarity (sentence embeddings). Everything gets cached to parquet so we never recompute unless we deliberately ask to.

A word on what we're featurizing: HH-RLHF stores full conversations, but we extracted just the final assistant response in the loader. That's what we're scoring here, which is what actually matters.

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd

from inference_lens.features.extractor import (
    compute_lexical_features,
    compute_embeddings,
    compute_rouge_l,
    extract_all_features,
)
from inference_lens.utils.logging import setup_logging

setup_logging()
pd.set_option('display.max_colwidth', 100)

## Load the flattened dataset

Picking up from where notebook 01 left off.

In [2]:
df = pd.read_parquet('../../data/processed/hh_rlhf_flat.parquet')
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
df.head(2)

Loaded 91,038 rows
Columns: ['chosen_response', 'rejected_response', 'chosen_full', 'rejected_full', 'subset', 'original_split']


,chosen_response,rejected_response,chosen_full,rejected_full,subset,original_split
0,A horseshoe is usually made out of metal and is about 3 to 3.5 inches long and around 1 inch thi...,"Horseshoes are either metal or plastic discs. The horseshoes come in different weights, and the ...","\n\nHuman: Hi, I want to learn to play horseshoes. Can you teach me?\n\nAssistant: I can, but ma...","\n\nHuman: Hi, I want to learn to play horseshoes. Can you teach me?\n\nAssistant: I can, but ma...",helpful-base,train
1,"Kids are great at fidgeting, so let’s make sure we give them permission to fidget.","Yeah, definitely. What works for you personally? What works for other people?",\n\nHuman: How do I teach kids to meditate?\n\nAssistant: Great question! That’s a really useful...,\n\nHuman: How do I teach kids to meditate?\n\nAssistant: Great question! That’s a really useful...,helpful-base,train


## Sanity check on response extraction

Before computing features on 170K rows, let's verify the assistant response extraction actually worked on a few examples.

In [3]:
sample = df.sample(3, random_state=0)

for i, row in sample.iterrows():
    print(f'--- row {i} ---')
    print(f'chosen_response  : {row["chosen_response"][:200]}')
    print(f'rejected_response: {row["rejected_response"][:200]}')
    print()

--- row 37995 ---
chosen_response  : You can start by learning about how to wear cowboy hats.  If you want to act like a cowboy, you might want to learn some cowboy songs and dances.  To actually be a cowboy, you might want to learn how 
rejected_response: Do you mean for a costume?  Or you want to be one in real life?

--- row 57726 ---
chosen_response  : Well, I don't want to tell you that.
rejected_response: Well I guess I could tell you to ask them to cry louder, so it's really obvious to everyone that they're annoying you.

--- row 63582 ---
chosen_response  : It’s a good idea to check with them, but keep in mind that they might be aware of her privacy preferences, and not want to share information about her contact information.  It might be better to go wi
rejected_response: I think you could try that.  Are there any friends you’re sure she’d be in contact with?  Maybe you could think of some names, and I could try calling them on your behalf, with an excuse to try to ask



## Dry run on 500 rows

Always run on a small slice before committing to the full dataset. Catches bugs cheaply.

In [4]:
sample_df = df.sample(500, random_state=42).copy()

# test lexical features first since they're fast
chosen_texts = sample_df['chosen_response'].tolist()
rejected_texts = sample_df['rejected_response'].tolist()

lex = compute_lexical_features(chosen_texts)
print('Lexical features shape:', lex.shape)
lex.describe().round(2)

Lexical features: 100%|██████████| 500/500 [00:00<00:00, 1000.45it/s]

Lexical features shape: (500, 3)


,token_length,type_token_ratio,flesch_score
count,500.00,500.00,500.00
mean,38.62,0.88,75.73
std,41.19,0.12,19.99
min,1.00,0.22,-14.81
25%,13.00,0.81,63.65
50%,26.00,0.90,77.15
75%,48.00,1.00,88.90
max,296.00,1.00,121.22


In [5]:
# ROUGE-L on the sample
# reference is chosen_response itself since we're measuring how much the chosen
# response overlaps with the rejected one -- informative for understanding divergence
rouge_scores = compute_rouge_l(rejected_texts, chosen_texts)
print(f'ROUGE-L (rejected vs chosen) -- mean: {np.mean(rouge_scores):.3f}  median: {np.median(rouge_scores):.3f}')

2026-05-15 18:58:35  INFO      absl  Using default tokenizer.


ROUGE-L: 100%|██████████| 500/500 [00:00<00:00, 2169.39it/s]

ROUGE-L (rejected vs chosen) -- mean: 0.138  median: 0.128


In [6]:
# embeddings on the sample -- takes a minute or so
emb = compute_embeddings(chosen_texts[:100])
print(f'Embeddings shape: {emb.shape}  dtype: {emb.dtype}')
print(f'Norm check (should be ~1.0): {np.linalg.norm(emb[0]):.4f}')

2026-05-15 18:58:39  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-05-15 18:58:39  WARNING   huggingface_hub.utils._http  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-15 18:58:39  INFO      httpx  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-15 18:58:39  INFO      httpx  HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-05-15 18:58:39  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-15 18:58:39  INFO      httpx  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-15 18:58:39  INFO      httpx  HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

2026-05-15 18:58:44  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-05-15 18:58:44  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-15 18:58:44  INFO      httpx  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-05-15 18:58:44  INFO      httpx  HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-05-15 18:58:44  INFO      httpx  HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-05-15 18:58:44  INFO      inference_lens.features.extractor  Encoding 100 texts with all-MiniLM-L6-v2 (batch_size=64)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (100, 384)  dtype: float32
Norm check (should be ~1.0): 1.0000


## Full feature extraction

Now we run on the complete dataset. This will take a while on first run (embeddings are the slow part). Subsequent runs skip straight to loading from cache.

We run feature extraction separately on chosen and rejected responses, then concatenate with a label column.

In [7]:
import os
os.makedirs('../../data/processed', exist_ok=True)

# build a long-form dataframe: each row is one response with a label
# chosen = 1, rejected = 0
chosen_df = df[['chosen_response', 'subset', 'original_split']].copy()
chosen_df = chosen_df.rename(columns={'chosen_response': 'response'})
chosen_df['label'] = 1

rejected_df = df[['rejected_response', 'subset', 'original_split']].copy()
rejected_df = rejected_df.rename(columns={'rejected_response': 'response'})
rejected_df['label'] = 0

long_df = pd.concat([chosen_df, rejected_df], ignore_index=True)
long_df = long_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f'Long-form dataset: {len(long_df):,} rows')
print(f'Label balance: {long_df["label"].value_counts().to_dict()}')

Long-form dataset: 182,076 rows
Label balance: {0: 91038, 1: 91038}


In [8]:
# save long-form before features so we have a clean checkpoint
long_df.to_parquet('../../data/processed/hh_rlhf_long.parquet', index=False)
print('Saved hh_rlhf_long.parquet')

Saved hh_rlhf_long.parquet


In [9]:
# compute all features and cache
# pass force_recompute=True if you change the feature set and need a fresh cache
features_df, embeddings = extract_all_features(
    long_df,
    text_col='response',
    reference_col='response',
    include_bertscore=False,   # too slow without GPU, skip for now
    features_cache='../../data/processed/features.parquet',
    embeddings_cache='../../data/processed/embeddings.npy',
    force_recompute=False,
)

print(f'Features shape: {features_df.shape}')
print(f'Embeddings shape: {embeddings.shape}')
features_df.describe().round(3)

2026-05-15 18:58:47  INFO      inference_lens.features.extractor  Computing features for 182076 responses


Lexical features: 100%|██████████| 182076/182076 [00:07<00:00, 23579.99it/s]

2026-05-15 18:58:54  INFO      absl  Using default tokenizer.



ROUGE-L: 100%|██████████| 182076/182076 [01:40<00:00, 1816.19it/s]


2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "H

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-15 19:00:35  INFO      httpx  HTTP Request: HEAD https://huggingface.co/a

Batches:   0%|          | 0/2845 [00:00<?, ?it/s]

2026-05-15 19:03:07  INFO      inference_lens.features.extractor  Cached features to ../../data/processed/features.parquet and embeddings to ../../data/processed/embeddings.npy
Features shape: (182076, 4)
Embeddings shape: (182076, 384)


,token_length,type_token_ratio,flesch_score,rouge_l
count,182076.000,182076.000,182076.000,182076.000
mean,36.542,0.878,75.279,0.999
std,36.005,0.117,26.156,0.036
min,0.000,0.000,-3293.010,0.000
25%,13.000,0.800,63.601,1.000
50%,25.000,0.900,76.555,1.000
75%,48.000,1.000,88.905,1.000
max,646.000,1.000,121.220,1.000


## Attach labels and save final feature table

In [10]:
features_df['label'] = long_df['label'].values
features_df['subset'] = long_df['subset'].values
features_df['original_split'] = long_df['original_split'].values

features_df.to_parquet('../../data/processed/features_with_labels.parquet', index=False)
print(f'Saved features_with_labels.parquet -- {len(features_df):,} rows, {features_df.shape[1]} columns')
features_df.head(3)

Saved features_with_labels.parquet -- 182,076 rows, 7 columns


,token_length,type_token_ratio,flesch_score,rouge_l,label,subset,original_split
0,12,0.916667,88.905,1.0,0,harmless-base,train
1,5,1.000000,32.560,1.0,0,harmless-base,train
2,9,0.888889,103.700,1.0,1,helpful-base,train


## Quick check: do features differ between chosen and rejected?

If our features have zero signal, the models have nothing to learn from. Let's see if the distributions look meaningfully different before moving to EDA.

In [11]:
feature_cols = ['token_length', 'type_token_ratio', 'flesch_score', 'rouge_l']

print(f'{"Feature":<22} {"Chosen mean":>14} {"Rejected mean":>14} {"Diff":>10}')
print('-' * 64)
for col in feature_cols:
    chosen_mean = features_df[features_df['label'] == 1][col].mean()
    rejected_mean = features_df[features_df['label'] == 0][col].mean()
    diff = chosen_mean - rejected_mean
    print(f'{col:<22} {chosen_mean:>14.3f} {rejected_mean:>14.3f} {diff:>+10.3f}')

Feature                   Chosen mean  Rejected mean       Diff
----------------------------------------------------------------
token_length                   36.976         36.107     +0.868
type_token_ratio                0.876          0.880     -0.003
flesch_score                   74.975         75.583     -0.609
rouge_l                         0.999          0.999     -0.000
